# Lab 5 — The agent loop, by hand

**~55 minutes.** Ed opens his *Agentic AI* course by building an agent with no framework at
all, and it is the right call: once you have written the loop, every framework becomes
"someone else's version of this".

You will build a research assistant over this repository, then add the guardrails that
separate a demo from something you would let near production: a step cap, loop detection,
and an approval gate on the one tool that touches the outside world.

In [ ]:
import json, time
from pathlib import Path

from shared import client, model_name

REPO = Path("../..").resolve()
TRACE = []          # every step, for the debrief at the end


def list_files(subdir: str = ".") -> str:
    """List files under a repo-relative directory."""
    target = (REPO / subdir).resolve()
    if not str(target).startswith(str(REPO)):          # path traversal guard
        return json.dumps({"error": "path outside the repository"})
    if not target.exists():
        return json.dumps({"error": f"no such directory: {subdir}"})
    names = sorted(p.name + ("/" if p.is_dir() else "") for p in target.iterdir())
    return json.dumps({"dir": subdir, "entries": names[:80]})


def read_file(path: str, max_chars: int = 4000) -> str:
    """Read a repo-relative text file."""
    target = (REPO / path).resolve()
    if not str(target).startswith(str(REPO)):
        return json.dumps({"error": "path outside the repository"})
    if not target.is_file():
        return json.dumps({"error": f"not a file: {path}. Use list_files first."})
    body = target.read_text(errors="replace")[:max_chars]
    return json.dumps({"path": path, "chars": len(body), "content": body})


def send_report(title: str, body: str) -> str:
    """Deliver a finished report. THIS IS THE DANGEROUS ONE — it leaves the sandbox."""
    print(f"\n>>> SENDING REPORT: {title}\n{body[:500]}\n")
    return json.dumps({"sent": True, "title": title})


print(list_files(".")[:300])

In [ ]:
def schema(name, description, props, required):
    return {"type": "function", "function": {"name": name, "description": description,
            "parameters": {"type": "object", "properties": props, "required": required}}}


TOOLS = [
    schema("list_files", "List files and directories under a repository-relative path. "
           "Use this before guessing a filename.",
           {"subdir": {"type": "string", "description": "e.g. 'handson-lab/notebooks'"}}, []),
    schema("read_file", "Read a repository-relative text file. Returns up to 4000 characters.",
           {"path": {"type": "string"}}, ["path"]),
    schema("send_report", "Deliver the finished report to the user. Call this exactly once, "
           "at the very end, when the task is complete.",
           {"title": {"type": "string"}, "body": {"type": "string"}}, ["title", "body"]),
]

IMPL = {"list_files": list_files, "read_file": read_file, "send_report": send_report}
NEEDS_APPROVAL = {"send_report"}

## 1. The loop

Perceive, reason, act, repeat. Fifteen lines. Everything else in this notebook is a
guardrail wrapped around it.

In [ ]:
SYSTEM = """You are a research assistant working inside a git repository.

Explore with list_files and read_file before drawing conclusions. Never guess at the
contents of a file you have not read. Cite file paths in your findings.

When you have enough to answer, call send_report exactly once with the finished text."""


def run_agent(task: str, max_steps: int = 12) -> str:
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": task}]

    for step in range(max_steps):
        reply = client().chat.completions.create(
            model=model_name(), messages=messages, tools=TOOLS, temperature=0.1,
        ).choices[0].message

        if not reply.tool_calls:
            TRACE.append({"step": step, "type": "final"})
            return reply.content or "(empty answer)"

        messages.append(reply)
        for call in reply.tool_calls:
            name, raw_args = call.function.name, call.function.arguments or "{}"
            print(f"  step {step}: {name}({raw_args[:90]})")
            TRACE.append({"step": step, "tool": name, "args": raw_args})
            try:
                result = IMPL[name](**json.loads(raw_args))
            except Exception as exc:
                result = json.dumps({"error": f"{type(exc).__name__}: {exc}"})
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

    return f"Stopped: hit the {max_steps}-step cap without finishing."


print(run_agent("What does this repository contain? Summarise it in five bullet points."))

## 2. Guardrails

Three failure modes, three controls. Add them to your loop:

1. **Loops** — the agent calls the same tool with the same arguments forever. Detect a
   repeat and feed the error back to the model instead of executing it again.
2. **Runaway cost** — the step cap you already have, plus a wall-clock timeout.
3. **Irreversible action** — `send_report` leaves the sandbox. Require a human `y` before it
   runs. In a real system this is the control that actually holds.

In [ ]:
def run_agent_guarded(task: str, max_steps: int = 12, timeout_s: int = 120) -> str:
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": task}]
    seen, deadline = set(), time.time() + timeout_s

    for step in range(max_steps):
        if time.time() > deadline:
            return "Stopped: wall-clock timeout."

        reply = client().chat.completions.create(
            model=model_name(), messages=messages, tools=TOOLS, temperature=0.1,
        ).choices[0].message

        if not reply.tool_calls:
            return reply.content or "(empty answer)"

        messages.append(reply)
        for call in reply.tool_calls:
            name, raw_args = call.function.name, call.function.arguments or "{}"
            fingerprint = (name, raw_args)

            if fingerprint in seen:                      # 1. loop detection
                result = json.dumps({"error": "You already made this exact call and got a "
                                              "result. Use it, or try something different."})
            elif name in NEEDS_APPROVAL:                 # 3. human in the loop
                print(f"\n  APPROVAL NEEDED: {name}({raw_args[:200]})")
                if input("  run it? [y/N] ").strip().lower() != "y":
                    result = json.dumps({"error": "The human declined this action."})
                else:
                    result = IMPL[name](**json.loads(raw_args))
            else:
                print(f"  step {step}: {name}({raw_args[:90]})")
                try:
                    result = IMPL[name](**json.loads(raw_args))
                except Exception as exc:
                    result = json.dumps({"error": f"{type(exc).__name__}: {exc}"})

            seen.add(fingerprint)
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

    return f"Stopped: hit the {max_steps}-step cap without finishing."


print(run_agent_guarded("Read the hands-on lab README and report what a newcomer must "
                        "install before Lab 4."))

## 3. Make it fail

Deliberately break it and watch what happens — this is the most valuable ten minutes of the
workshop.

- Ask for something the repository does not contain ("summarise the Terraform modules").
  Does it admit the gap, or invent one?
- Set `max_steps=3` on a task that needs more. Does it fail loudly or claim success?
- Remove `list_files` from `TOOLS` and re-run. Watch it guess at filenames.

In [ ]:
print(run_agent_guarded("Summarise the Terraform modules in this repository.", max_steps=6))
print("\n" + "=" * 70 + "\n")
print(run_agent_guarded("Compare every document in the repo and rank them by usefulness.",
                        max_steps=3))

## Stretch goals

1. **Your digital twin.** Ed's Week 1 project: load your own CV as a text file, give the
   agent a `record_unknown_question` tool, and let it answer questions about you — escalating
   anything it cannot support from the document. Same loop, different tools.
2. **A verification step.** After the agent reports done, run a second call that checks the
   claim against the tool trace. "Silent success" is the failure mode teams find last.
3. **Print the trace.** Dump `TRACE` as a table: step, tool, arguments. This is what an
   observability tool shows you, and building it once explains why you want one.
4. **Swap the loop for a framework.** Rebuild this in the OpenAI Agents SDK or CrewAI, as
   Ed's course does in Weeks 2-3, and compare the line counts and the debuggability.